# 10-Drone UAV Swarm MARL Training on Google Colab (Pure-PyTorch MAPPO, Contract v2)
Follows **PROJECT_PLAN.md §17.4**. Always execute Cell 4 (mandatory parity & 10-drone smoke test) before starting any curriculum stage.

In [ ]:
# Cell 1 — Mount Google Drive & Inspect Runtime
from google.colab import drive
drive.mount("/content/drive")

import os, psutil, torch
RUN_ROOT = "/content/drive/MyDrive/uav_swarm_marl/runs"
os.makedirs(RUN_ROOT, exist_ok=True)
print("CPUs:", os.cpu_count(),
      "| RAM GB:", round(psutil.virtual_memory().total / 1e9, 1),
      "| CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU-only")

In [ ]:
%%bash
# Cell 2 — Clone or Pull the Repository
REPO_URL="https://github.com/janistall/SEM3_MAINEL_PLSWORK.git"
BRANCH="main"

if [ ! -d /content/uav_EL ]; then
  git clone --branch "$BRANCH" "$REPO_URL" /content/uav_EL
else
  cd /content/uav_EL && git fetch origin && git checkout "$BRANCH" && git pull
fi
cd /content/uav_EL && git log -n 3 --oneline

In [ ]:
%%bash
# Cell 3 — Install Lightweight Dependencies & Verify Versions (No Ray RLlib!)
cd /content/uav_EL
pip install -q -r requirements-colab.txt
python3 -c "import torch, gymnasium, pettingzoo, numpy; print('torch', torch.__version__, 'numpy', numpy.__version__, 'pettingzoo', pettingzoo.__version__)"

In [ ]:
%%bash
# Cell 4 — Mandatory Parity & 10-Drone Smoke Test (DO NOT SKIP)
cd /content/uav_EL
PYTHONPATH=src/swarm_control python3 -m unittest discover -s src/swarm_control/test -v

python3 - <<'PY'
import numpy as np, torch
from marl_training.envs import DroneSwarmEnv
from marl_training.models import SwarmActorNet, CentralizedValueNet

env = DroneSwarmEnv(num_drones=10, max_steps=600, shield=True)
obs, _ = env.reset(seed=7)
assert len(obs) == 10, f"expected 10 drones, got {len(obs)}"
sample = next(iter(obs.values()))
assert sample.shape == (62,), f"expected 62-D v2 obs, got {sample.shape}"

actor = SwarmActorNet(obs_dim=62, act_dim=4)
critic = CentralizedValueNet(num_drones=10, obs_dim=62)
obs_t = torch.as_tensor(np.stack([obs[a] for a in env.agents]), dtype=torch.float32)
act_mean = actor.deterministic(obs_t)
assert act_mean.shape == (10, 4)

for _ in range(20):
    actions = {a: env.action_space(a).sample() for a in env.agents}
    obs, rew, term, trunc, info = env.step(actions)
    if not env.agents:
        break
print("SMOKE TEST PASSED: 10 drones, obs_dim=62, MAPPO actor/critic OK, finite rewards =",
      all(np.isfinite(list(rew.values()))))
PY

In [ ]:
# Cell 5 — Launch 4-Stage MAPPO Curriculum Training (Always at N = 10)
# Uses `!python3 -u` (not `%%bash`) so Colab streams every iteration line live.
%cd /content/uav_EL
!python3 -u -m marl_training.train \
  --num-drones 10 \
  --max-steps 600 \
  --stage stage1_open \
  --iters 150 \
  --rollout-steps 2400 \
  --ppo-epochs 6 \
  --minibatch-size 1024 \
  --lr 3e-4 \
  --eval-every 10 \
  --checkpoint-every 10 \
  --seed 7 \
  --out /content/drive/MyDrive/uav_swarm_marl/runs/exp01_stage1_open

In [ ]:
# Cell 6 — Recovering & Resuming After a Colab Disconnect
%cd /content/uav_EL
!python3 -u -m marl_training.train \
  --num-drones 10 --max-steps 600 --stage stage1_open --iters 150 \
  --rollout-steps 2400 --eval-every 10 --checkpoint-every 10 --seed 7 \
  --resume /content/drive/MyDrive/uav_swarm_marl/runs/exp01_stage1_open/train_state_latest.pt \
  --out /content/drive/MyDrive/uav_swarm_marl/runs/exp01_stage1_open

In [ ]:
# Cell 7 — Plot Learning Curves & Diagnose MAPPO Training Health
import json, pathlib, pandas as pd, matplotlib.pyplot as plt

RUN_DIR = pathlib.Path("/content/drive/MyDrive/uav_swarm_marl/runs/exp01_stage1_open")
rows = [json.loads(line) for line in (RUN_DIR / "metrics.jsonl").read_text().splitlines() if line.strip()]
df = pd.DataFrame(rows)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
df.plot(x="env_steps_total", y="episode_reward_mean", ax=axes[0, 0], title="Train Episode Reward")
if "eval_success_rate" in df:
    df.dropna(subset=["eval_success_rate"]).plot(x="env_steps_total", y="eval_success_rate", ax=axes[0, 1], title="Eval Success Rate (L+S)")
if "eval_mean_shield_intervention" in df:
    df.dropna(subset=["eval_mean_shield_intervention"]).plot(x="env_steps_total", y="eval_mean_shield_intervention", ax=axes[1, 0], title="Mean Shield Intervention (m/s)")
df.plot(x="env_steps_total", y=["value_loss", "approx_kl"], ax=axes[1, 1], title="Critic Value Loss & Approx KL")
plt.tight_layout(); plt.show()

In [ ]:
# Cell 8 — Run Pre-Download Shield-ON vs. Shield-OFF Ablation in Colab
# For Stage 1 (0 static obstacles), evaluate with `--stage stage1_open` (or `--modes open`).
# For Stage 2–4 checkpoints, switch to `--modes clean rain birds clutter target_change`.
%cd /content/uav_EL
!python3 -u -m marl_training.eval \
  --checkpoint /content/drive/MyDrive/uav_swarm_marl/runs/exp01_stage1_open/swarm_policy_best.pt \
  --num-drones 10 --episodes 20 --seeds 11 30 --stage stage1_open
# Full obstacle/robustness ablation (use after Stage 2–4 training):
# !python3 -u -m marl_training.eval \
#   --checkpoint /content/drive/MyDrive/uav_swarm_marl/runs/exp04_stage4_domain_rand/swarm_policy_best.pt \
#   --num-drones 10 --episodes 20 --seeds 11 30 --modes clean rain birds clutter target_change

In [ ]:
# Cell 9 — Export to Google Drive /exports & Verify ActorPolicy Load
import sys, shutil, pathlib, numpy as np
sys.path.insert(0, "/content/uav_EL/src/swarm_control")
from swarm_control.contract import ActorPolicy

src = pathlib.Path("/content/drive/MyDrive/uav_swarm_marl/runs/exp01_stage1_open/swarm_policy_best.pt")
dst = pathlib.Path("/content/drive/MyDrive/uav_swarm_marl/exports/swarm_policy_best.pt")
dst.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(src, dst)

p = ActorPolicy(dst, actor_index=0)
assert p.contract.schema == "swarm-coop-enu-v2" and p.contract.obs_dim == 62 and p.contract.agents == 10
out = p(np.zeros(62, dtype="float32"))
assert out.shape == (4,) and np.all(np.isfinite(out))
print("EXPORTED & VERIFIED OK:", dst, "zero-obs action =", out)